In [1]:
import os
os.system('pip install --quiet lightgbm')                          # [NEW] install lightgbm
 
print("--- Step 1: Importing libraries ---")
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import timm
import torchvision.transforms as T
from PIL import Image
import numpy as np
from tqdm import tqdm
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import shutil
import requests
import time
import lightgbm as lgb                                             # import lightgbm
from sklearn.decomposition import PCA                              # [NEW] reduce 2048 embedding dims → less overfitting

--- Step 1: Importing libraries ---


In [2]:
# --- Step 2: Configuration for the Kaggle Environment ---
 
print("\n--- Step 2: Setting up configuration ---")
class KaggleConfig:
    
    KAGGLE_INPUT_DIR = Path("/kaggle/input/datasets/bboyattitude/mavericks-amazon-ml-dataset")
    
    KAGGLE_WORKING_DIR = Path("/kaggle/working/")
    IMAGE_DIR = KAGGLE_WORKING_DIR / "images"
    TRAIN_CSV = KAGGLE_INPUT_DIR / "train.csv"
    TEST_CSV = KAGGLE_INPUT_DIR / "test.csv"
 
    # --- Model ---
    TEXT_MODEL_NAME = "distilbert-base-uncased"
    IMAGE_MODEL_NAME = "efficientnet_b0"
    IMG_SIZE = 224
 
    # --- Training 
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    BATCH_SIZE = 64
    EPOCHS = 10
    LEARNING_RATE = 1e-4
    NUM_WORKERS = 2
    
    # --- Image Download ---
    DOWNLOAD_WORKERS = 128
 
    # --- PCA ---
    PCA_COMPONENTS = 300           # [NEW] 2048 dims → 300, removes noise dims LightGBM was memorizing
 
    # --- LightGBM ---
    LGBM_PARAMS = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.03,     # was 0.05 → slower learning, less overfit
        'num_leaves': 63,          # was 127 → simpler trees, can't memorize
        'min_data_in_leaf': 50,    # [NEW] each leaf needs 50+ samples, no single-product memorization
        'feature_fraction': 0.6,   # was 0.8 → each tree sees 60% of dims, like dropout
        'bagging_fraction': 0.7,   # was 0.8 → each tree sees 70% of rows
        'bagging_freq': 5,
        'lambda_l1': 0.1,          # [NEW] L1 reg, kills weak/noisy features
        'lambda_l2': 0.1,          # [NEW] L2 reg, shrinks over-reliance on any single dim
        'verbose': -1,
        'n_jobs': -1,
    }
    LGBM_NUM_ROUNDS = 2000         # was 1000 → more rounds to compensate for slower lr
    LGBM_EARLY_STOPPING = 100      # was 50 → more patience
 
# Instantiate the config
config = KaggleConfig()
 
print(f"Device: {config.DEVICE}")
print(f"Train CSV path: {config.TRAIN_CSV}")
print(f"Image directory will be: {config.IMAGE_DIR}")
 


--- Step 2: Setting up configuration ---
Device: cuda
Train CSV path: /kaggle/input/datasets/bboyattitude/mavericks-amazon-ml-dataset/train.csv
Image directory will be: /kaggle/working/images


In [3]:
# --- Step 3: High-Speed Image Downloader (with retries) ---
 
print("\n--- Step 3: Defining the image downloader ---")
def download_one_image(args):
    img_url, img_path = args
    if img_path.exists():
        return "skipped"
    # Retry logic
    for attempt in range(3):
        try:
            response = requests.get(img_url, timeout=20)
            response.raise_for_status()
            with open(img_path, 'wb') as f:
                f.write(response.content)
            return "downloaded"
        except requests.exceptions.RequestException:
            time.sleep(1 + attempt) # Simple backoff
            if attempt == 2: return "failed"
 
def download_image_set(csv_path, image_root_dir, set_name, max_workers=16):
    final_save_dir = image_root_dir / set_name
    final_save_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(csv_path)
    tasks = [(row['image_link'], final_save_dir / f"{row['sample_id']}.jpg") for _, row in df.iterrows()]
    print(f"Downloading {len(tasks)} images for '{set_name}' set with {max_workers} workers...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(download_one_image, tasks), total=len(tasks)))
    print(f"Finished '{set_name}' set download. Summary: Skipped={results.count('skipped')}, Downloaded={results.count('downloaded')}, Failed={results.count('failed')}")


--- Step 3: Defining the image downloader ---


In [4]:
# --- Step 4: Data Processing (Dataset Class) ---
 
print("\n--- Step 4: Defining the Dataset class ---")
class ProductDataset(Dataset):
    def __init__(self, csv_file, image_dir, tokenizer, is_train=True):
        self.df = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.is_train = is_train
        self.tokenizer = tokenizer
        self.transform = T.Compose([
            T.Resize((config.IMG_SIZE, config.IMG_SIZE)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['catalog_content'])
        img_folder = 'train' if self.is_train else 'test'
        img_path = self.image_dir / img_folder / f"{row['sample_id']}.jpg"
 
        try:
            image = Image.open(img_path).convert("RGB")
            image = self.transform(image)
        except (FileNotFoundError, OSError):
            image = torch.zeros((3, config.IMG_SIZE, config.IMG_SIZE))
 
        inputs = self.tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
        input_ids = inputs['input_ids'].squeeze(0)
        attention_mask = inputs['attention_mask'].squeeze(0)
 
        if self.is_train:
            price = float(row['price'])
            target = torch.tensor(np.log1p(price), dtype=torch.float32)
            return {'input_ids': input_ids, 'attention_mask': attention_mask, 'image': image, 'target': target}
        else:
            return {'input_ids': input_ids, 'attention_mask': attention_mask, 'image': image}
 
 


--- Step 4: Defining the Dataset class ---


In [5]:
# --- Step 5: Model Architecture ---
 
print("\n--- Step 5: Defining the Model architecture ---")
class MultiModalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_model = AutoModel.from_pretrained(config.TEXT_MODEL_NAME)
        self.image_model = timm.create_model(config.IMAGE_MODEL_NAME, pretrained=True, num_classes=0)
        text_features = self.text_model.config.hidden_size
        image_features = self.image_model.num_features
        self.head = nn.Sequential(nn.Linear(text_features + image_features, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, 1))
 
    def forward(self, input_ids, attention_mask, image):
        text_output = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_embedding = text_output.last_hidden_state[:, 0, :]
        image_embedding = self.image_model(image)
        combined_features = torch.cat([text_embedding, image_embedding], dim=1)
        output = self.head(combined_features)
        return output
 
    # [NEW] extract combined embeddings WITHOUT passing through the dense head
    def extract_embeddings(self, input_ids, attention_mask, image):
        text_output = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_embedding = text_output.last_hidden_state[:, 0, :]
        image_embedding = self.image_model(image)
        combined_features = torch.cat([text_embedding, image_embedding], dim=1)
        return combined_features                                    # shape: (batch, text_dim + image_dim)
 
 
# [NEW] Helper: run a DataLoader through the frozen NN and collect embeddings + targets
def extract_embeddings_from_loader(model, loader, device, is_train=True):
    model.eval()
    all_embeddings = []
    all_targets = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting embeddings"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            image = batch['image'].to(device)
            embeddings = model.extract_embeddings(input_ids, attention_mask, image)
            all_embeddings.append(embeddings.cpu().numpy())
            if is_train:
                all_targets.append(batch['target'].numpy())
    X = np.concatenate(all_embeddings, axis=0)
    y = np.concatenate(all_targets, axis=0) if is_train else None
    return X, y


--- Step 5: Defining the Model architecture ---


In [6]:
# --- Step 6: The Main Execution Block (DISK SPACE EFFICIENT) ---
 
if __name__ == '__main__':
    # === STAGE 1: Train NN end-to-end with Dense Head ===
    print("\n--- Step 6.1: Starting TRAINING image download ---")
    download_image_set(config.TRAIN_CSV, config.IMAGE_DIR, 'train', max_workers=config.DOWNLOAD_WORKERS)
    
    print("\n--- Step 6.2: Preparing for training ---")
    tokenizer = AutoTokenizer.from_pretrained(config.TEXT_MODEL_NAME)
    train_dataset = ProductDataset(config.TRAIN_CSV, config.IMAGE_DIR, tokenizer, is_train=True)
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS)
 
    device = torch.device(config.DEVICE)
    model = MultiModalModel().to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE)
 
    print("\n--- Step 6.3: STAGE 1 — Training NN with dense head to make embeddings price-aware ---")
    model.train()
    for epoch in range(config.EPOCHS):
        loop = tqdm(train_loader, leave=True)
        total_loss = 0
        for batch in loop:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            image = batch['image'].to(device)
            target = batch['target'].to(device)
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask, image)
            loss = criterion(outputs.squeeze(-1), target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            loop.set_description(f"Epoch {epoch+1}/{config.EPOCHS}")
            loop.set_postfix(loss=loss.item())
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")
 
    # === STAGE 2: Freeze NN, extract embeddings, train LightGBM ===  # [NEW]
    print("\n--- Step 6.4: STAGE 2 — Freezing NN and extracting train embeddings ---")
    # Freeze all NN weights — only LightGBM will train from here
    for param in model.parameters():
        param.requires_grad = False
 
    # Need a non-shuffled loader so embeddings align with targets correctly
    embed_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)
    X_train, y_train = extract_embeddings_from_loader(model, embed_loader, device, is_train=True)
    print(f"Train embeddings shape: {X_train.shape}, Targets shape: {y_train.shape}")
 
    print("\n--- Step 6.5: STAGE 2 — Reducing embedding dims with PCA ---")
    # Root cause of overfitting: LightGBM finds spurious patterns in 2048 noisy dims
    # PCA keeps only the 300 most meaningful directions and discards the rest
    pca = PCA(n_components=config.PCA_COMPONENTS, random_state=42)
    X_train_pca = pca.fit_transform(X_train)
    print(f"Embeddings reduced: {X_train.shape[1]} dims → {X_train_pca.shape[1]} dims")
    print(f"Variance retained: {pca.explained_variance_ratio_.sum()*100:.1f}%")
 
    print("\n--- Step 6.6: STAGE 2 — Training LightGBM on PCA embeddings ---")
    # Split 90/10 for early stopping validation
    split_idx = int(len(X_train_pca) * 0.9)
    X_tr, X_val = X_train_pca[:split_idx], X_train_pca[split_idx:]
    y_tr, y_val = y_train[:split_idx],     y_train[split_idx:]
 
    lgb_train = lgb.Dataset(X_tr, label=y_tr)
    lgb_val   = lgb.Dataset(X_val, label=y_val, reference=lgb_train)
 
    lgbm_model = lgb.train(
        config.LGBM_PARAMS,
        lgb_train,
        num_boost_round=config.LGBM_NUM_ROUNDS,
        valid_sets=[lgb_train, lgb_val],
        valid_names=['train', 'valid'],
        callbacks=[
            lgb.early_stopping(config.LGBM_EARLY_STOPPING, verbose=True),
            lgb.log_evaluation(period=50)
        ]
    )
    print(f"LightGBM training done. Best iteration: {lgbm_model.best_iteration}")


--- Step 6.1: Starting TRAINING image download ---


100%|██████████| 75000/75000 [07:18<00:00, 170.96it/s]


Finished 'train' set download. Summary: Skipped=0, Downloaded=74999, Failed=1

--- Step 6.2: Preparing for training ---


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

Epoch 9 Average Loss: 0.0902


Epoch 10/10: 100%|██████████| 1172/1172 [31:50<00:00,  1.63s/it, loss=0.0718]


Epoch 10 Average Loss: 0.0807

--- Step 6.4: STAGE 2 — Freezing NN and extracting train embeddings ---


Extracting embeddings: 100%|██████████| 1172/1172 [31:20<00:00,  1.60s/it]


Train embeddings shape: (75000, 2048), Targets shape: (75000,)

--- Step 6.5: STAGE 2 — Reducing embedding dims with PCA ---
Embeddings reduced: 2048 dims → 300 dims
Variance retained: 75.2%

--- Step 6.6: STAGE 2 — Training LightGBM on PCA embeddings ---
Training until validation scores don't improve for 100 rounds
[50]	train's rmse: 0.395334	valid's rmse: 0.401698
[100]	train's rmse: 0.264571	valid's rmse: 0.274933
[150]	train's rmse: 0.228909	valid's rmse: 0.242869
[200]	train's rmse: 0.21699	valid's rmse: 0.235194
[250]	train's rmse: 0.208872	valid's rmse: 0.23144
[300]	train's rmse: 0.202126	valid's rmse: 0.229169
[350]	train's rmse: 0.196177	valid's rmse: 0.227367
[400]	train's rmse: 0.190728	valid's rmse: 0.225911
[450]	train's rmse: 0.185784	valid's rmse: 0.224801
[500]	train's rmse: 0.181241	valid's rmse: 0.223844
[550]	train's rmse: 0.176972	valid's rmse: 0.223042
[600]	train's rmse: 0.172986	valid's rmse: 0.222408
[650]	train's rmse: 0.16912	valid's rmse: 0.221754
[700]	trai

In [7]:
# LightGBM training completes
import pickle
import torch

# Save LightGBM
lgbm_model.save_model('/kaggle/working/lgbm_model.txt')

# Save NN
torch.save(model.state_dict(), '/kaggle/working/nn_model.pt')

# Save PCA
with open('/kaggle/working/pca.pkl', 'wb') as f:
    pickle.dump(pca, f)

print("All models saved!")

All models saved!


In [8]:
import os
# Find exact path first
print(os.listdir('/kaggle/input/datasets/bboyattitude/ecommerce-models'))

['nn_model.pt', 'lgbm_model.txt', 'pca.pkl']


In [9]:
import pickle
import lightgbm as lgb
import numpy as np
import torch
import pandas as pd
from pathlib import Path
import shutil

# Load all saved models
lgbm_model = lgb.Booster(model_file='/kaggle/input/datasets/bboyattitude/ecommerce-models/lgbm_model.txt')

with open('/kaggle/input/datasets/bboyattitude/ecommerce-models/pca.pkl', 'rb') as f:
    pca = pickle.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiModalModel().to(device)
model.load_state_dict(torch.load('/kaggle/input/datasets/bboyattitude/ecommerce-models/nn_model.pt'))
model.eval()

tokenizer = AutoTokenizer.from_pretrained(config.TEXT_MODEL_NAME)
print("Models loaded!")

# --- SMAPE: re-extract train embeddings to get X_val and y_val back ---
train_dataset = ProductDataset(config.TRAIN_CSV, config.IMAGE_DIR, tokenizer, is_train=True)
embed_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)
X_train, y_train = extract_embeddings_from_loader(model, embed_loader, device, is_train=True)
X_train_pca = pca.transform(X_train)

# Recreate same 90/10 split
split_idx = int(len(X_train_pca) * 0.9)
X_val = X_train_pca[split_idx:]
y_val = y_train[split_idx:]

# Save so you never need to do this again
np.save('/kaggle/working/X_val.npy', X_val)
np.save('/kaggle/working/y_val.npy', y_val)

# Calculate SMAPE
val_log_preds = lgbm_model.predict(X_val, num_iteration=lgbm_model.best_iteration)
val_preds = np.expm1(val_log_preds)
val_actuals = np.expm1(y_val)
val_smape = np.mean(200 * np.abs(val_preds - val_actuals) / (np.abs(val_actuals) + np.abs(val_preds) + 1e-8))
print(f"\n>>> Validation SMAPE: {val_smape:.2f}% (previous was 46.1%) <<<")

# --- FREE DISK SPACE BEFORE TEST DOWNLOAD ---         
if (config.IMAGE_DIR / 'train').exists():
    shutil.rmtree(config.IMAGE_DIR / 'train')
    print("Train images deleted — disk space freed!")
else:
    print("Train images already deleted, skipping.")



# --- Test prediction and submission ---
download_image_set(config.TEST_CSV, config.IMAGE_DIR, 'test', max_workers=config.DOWNLOAD_WORKERS)

test_dataset = ProductDataset(config.TEST_CSV, config.IMAGE_DIR, tokenizer, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)
X_test, _ = extract_embeddings_from_loader(model, test_loader, device, is_train=False)
X_test = pca.transform(X_test)
print(f"Test embeddings shape after PCA: {X_test.shape}")

log_predictions = lgbm_model.predict(X_test, num_iteration=lgbm_model.best_iteration)
predictions = np.expm1(log_predictions)

df_test = pd.read_csv(config.TEST_CSV)
predictions = predictions[:len(df_test)]
predictions = np.maximum(0, predictions)
submission_df = pd.DataFrame({'sample_id': df_test['sample_id'], 'price': predictions})
submission_df.to_csv('/kaggle/working/submission.csv', index=False)

print("\n--- Pipeline complete! ---")
print(f"Submission saved at: /kaggle/working/submission.csv")
print("\nFirst 5 predictions:")
print(submission_df.head())

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded!


Extracting embeddings: 100%|██████████| 1172/1172 [27:20<00:00,  1.40s/it]



>>> Validation SMAPE: 19.41% (previous was 46.1%) <<<
Train images deleted — disk space freed!


100%|██████████| 75000/75000 [07:46<00:00, 160.81it/s]


Finished 'test' set download. Summary: Skipped=0, Downloaded=74999, Failed=1


Extracting embeddings: 100%|██████████| 1172/1172 [27:31<00:00,  1.41s/it]


Test embeddings shape after PCA: (75000, 300)

--- Pipeline complete! ---
Submission saved at: /kaggle/working/submission.csv

First 5 predictions:
   sample_id      price
0     100179  19.420024
1     245611  18.056849
2     146263  20.812404
3      95658   5.052298
4      36806  15.582288
